# Multi-Model Serving Tests

Start `main.ipynb` first (`http://127.0.0.1:8001`).

In [1]:
import httpx
import sys
from pathlib import Path

BASE = "http://127.0.0.1:8001"
QWEN = "qwen2.5-0.5b"
TINY = "tinyllama-1.1b"

# Paths from gitignored local_paths.json (never hard-code absolutes here)
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "multi_model_serving" else Path.cwd()))
from local_paths import load_local_paths, models_dir, model_path

cfg = load_local_paths()
NOTEBOOK_DIR = Path(cfg["MULTI_MODEL_NOTEBOOK_DIR"])
MODELS_DIR = models_dir(cfg)
QWEN_MODEL_PATH = model_path("QWEN_MODEL", cfg)
TINYLLAMA_MODEL_PATH = model_path("TINYLLAMA_MODEL", cfg)

print("BASE:", BASE)
print("MODELS_DIR exists:", MODELS_DIR.exists())
print("  Qwen exists:", QWEN_MODEL_PATH.exists())
print("  TinyLlama exists:", TINYLLAMA_MODEL_PATH.exists())

BASE: http://127.0.0.1:8001
NOTEBOOK_DIR: <MULTI_MODEL_NOTEBOOK_DIR>
MODELS_DIR:   <MODELS_DIR>
  Qwen:       <MODELS_DIR>/Qwen2.5-0.5B-Instruct
  TinyLlama:  <MODELS_DIR>/TinyLlama-1.1B-Chat-v1.0


## List models

In [3]:
with httpx.Client(trust_env=False, proxy=None) as client:
    response = client.get(f"{BASE}/models", timeout=30.0)

print(response.status_code)
data = response.json()
print("available:", list(data["available_models"].keys()))
print("models_dir:", data.get("models_dir"))
for mid, meta in data["available_models"].items():
    print(f"  {mid}: resolved={meta.get('resolved_path')}")
print("loaded:", data["loaded_models"])

200
available: ['qwen2.5-0.5b', 'tinyllama-1.1b']
models_dir: <MODELS_DIR>
  qwen2.5-0.5b: resolved=<MODELS_DIR>/Qwen2.5-0.5B-Instruct
  tinyllama-1.1b: resolved=<MODELS_DIR>/TinyLlama-1.1B-Chat-v1.0
loaded: {}


## Generate — Qwen2.5

In [8]:
with httpx.Client(trust_env=False, proxy=None) as client:
    response = client.post(
        f"{BASE}/generate",
        json={
            "model_id": QWEN,
            "prompt": "What is the capital of the United States? Answer in one sentence.",
            "max_new_tokens": 48,
        },
        timeout=180.0,
    )

print(response.status
_code)
print(response.text)  # use .text so empty/non-JSON 500s are visible
if response.headers.get("content-type", "").startswith("application/json") and response.content:
    print(response.json())

200
{"model_id":"qwen2.5-0.5b","model_name":"Qwen2.5-0.5B-Instruct","prompt":"What is the capital of the United States? Answer in one sentence.","generated_text":"The capital of the United States is Washington, D.C., which was established as the nation's first city in 1790 to serve as its seat of government and to be the official residence of the President of the United States."}
{'model_id': 'qwen2.5-0.5b', 'model_name': 'Qwen2.5-0.5B-Instruct', 'prompt': 'What is the capital of the United States? Answer in one sentence.', 'generated_text': "The capital of the United States is Washington, D.C., which was established as the nation's first city in 1790 to serve as its seat of government and to be the official residence of the President of the United States."}


## Generate TinyLlama

In [5]:
with httpx.Client(trust_env=False, proxy=None) as client:
    response = client.post(
        f"{BASE}/generate",
        json={
            "model_id": TINY,
            "prompt": "What is 2+2? Answer briefly.",
            "max_new_tokens": 32,
        },
        timeout=180.0,
    )

print(response.status_code)
print(response.text)
if response.headers.get("content-type", "").startswith("application/json") and response.content:
    print(response.json())

200
{"model_id":"tinyllama-1.1b","model_name":"TinyLlama-1.1B-Chat-v1.0","prompt":"What is 2+2? Answer briefly.","generated_text":"2 + 2 = 4 in simple terms."}
{'model_id': 'tinyllama-1.1b', 'model_name': 'TinyLlama-1.1B-Chat-v1.0', 'prompt': 'What is 2+2? Answer briefly.', 'generated_text': '2 + 2 = 4 in simple terms.'}


## Predict

In [6]:
with httpx.Client(trust_env=False, proxy=None) as client:
    response = client.post(
        f"{BASE}/predict",
        json={
            "model_id": QWEN,
            "input_data": {
                "prompt": "Explain KV cache in one short sentence.",
                "max_new_tokens": 48,
            },
        },
        timeout=180.0,
    )

print(response.status_code)
print(response.text)
if response.headers.get("content-type", "").startswith("application/json") and response.content:
    print(response.json())

200
{"model_id":"qwen2.5-0.5b","model_name":"Qwen2.5-0.5B-Instruct","prompt":"Explain KV cache in one short sentence.","generated_text":"A K-V (Key-Value) Cache is a data structure that stores and retrieves key-value pairs efficiently using a hash table or a linked list."}
{'model_id': 'qwen2.5-0.5b', 'model_name': 'Qwen2.5-0.5B-Instruct', 'prompt': 'Explain KV cache in one short sentence.', 'generated_text': 'A K-V (Key-Value) Cache is a data structure that stores and retrieves key-value pairs efficiently using a hash table or a linked list.'}


## Loaded models

In [7]:
with httpx.Client(trust_env=False, proxy=None) as client:
    response = client.get(f"{BASE}/models", timeout=30.0)

print(response.status_code)
print("loaded_models:", response.json()["loaded_models"])

200
loaded_models: {'tinyllama-1.1b': 'TinyLlama-1.1B-Chat-v1.0', 'qwen2.5-0.5b': 'Qwen2.5-0.5B-Instruct'}


## Concurrent requests

In [8]:
import asyncio
import time

jobs = [
    (QWEN, "What is attention? One sentence."),
    (TINY, "What is continuous batching? One sentence."),
    (QWEN, "What is speculative decoding? One sentence."),
    (TINY, "What is a transformer? One sentence."),
]


async def call_api(model_id: str, prompt: str):
    async with httpx.AsyncClient(trust_env=False, proxy=None) as client:
        start = time.perf_counter()
        r = await client.post(
            f"{BASE}/generate",
            json={"model_id": model_id, "prompt": prompt, "max_new_tokens": 40},
            timeout=300.0,
        )
        elapsed = time.perf_counter() - start
        return {
            "model_id": model_id,
            "prompt": prompt,
            "time_s": round(elapsed, 2),
            "status": r.status_code,
            "response": r.json() if r.status_code == 200 else r.text,
        }


results = await asyncio.gather(*[call_api(m, p) for m, p in jobs])

for r in results:
    print(r)

{'model_id': 'qwen2.5-0.5b', 'prompt': 'What is attention? One sentence.', 'time_s': 3.39, 'status': 200, 'response': {'model_id': 'qwen2.5-0.5b', 'model_name': 'Qwen2.5-0.5B-Instruct', 'prompt': 'What is attention? One sentence.', 'generated_text': 'Attention refers to the process of selectively focusing on relevant information in a rapidly changing environment, enabling individuals and machines to make more informed decisions and improve their performance.'}}
{'model_id': 'tinyllama-1.1b', 'prompt': 'What is continuous batching? One sentence.', 'time_s': 3.64, 'status': 200, 'response': {'model_id': 'tinyllama-1.1b', 'model_name': 'TinyLlama-1.1B-Chat-v1.0', 'prompt': 'What is continuous batching? One sentence.', 'generated_text': 'Continuous batching is a technique used in data processing that involves processing batches of data in a single operation rather than processing individual records one at a time. It allows for the efficient and effective'}}
{'model_id': 'qwen2.5-0.5b', 'pr

## Warm latency

In [9]:
import time

prompt = "Explain KV cache in 2 sentences."

with httpx.Client(trust_env=False, proxy=None) as client:
    for i in range(2):
        start = time.perf_counter()
        response = client.post(
            f"{BASE}/generate",
            json={"model_id": QWEN, "prompt": prompt, "max_new_tokens": 48},
            timeout=180.0,
        )
        elapsed = time.perf_counter() - start
        print(f"call {i + 1}: {round(elapsed, 3)}s  status={response.status_code}")
        if i == 1:
            print(response.json())

call 1: 1.477s  status=200
call 2: 1.446s  status=200
{'model_id': 'qwen2.5-0.5b', 'model_name': 'Qwen2.5-0.5B-Instruct', 'prompt': 'Explain KV cache in 2 sentences.', 'generated_text': 'A Key-Value (KV) Cache is an application layer storage that stores data in a non-volatile manner using hash or bitmaps as key-value pairs. It provides fast access to data and can improve the performance of applications by reducing disk'}


# Extra tests

## Invalid model id

In [10]:
with httpx.Client(trust_env=False, proxy=None) as client:
    response = client.post(
        f"{BASE}/generate",
        json={"model_id": "does-not-exist", "prompt": "hello", "max_new_tokens": 8},
        timeout=30.0,
    )

print("status:", response.status_code)  # expect 404
print(response.text)

status: 404
{"detail":"Model does-not-exist not found"}


## Empty prompt

In [11]:
with httpx.Client(trust_env=False, proxy=None) as client:
    response = client.post(
        f"{BASE}/generate",
        json={"model_id": QWEN, "prompt": "", "max_new_tokens": 8},
        timeout=60.0,
    )

print("status:", response.status_code)  # expect 500 with Empty prompt
print(response.text)

status: 500
{"detail":{"error":"Empty prompt","traceback":"Traceback (most recent call last):\n  File \"<MULTI_MODEL_NOTEBOOK_DIR>/app/server.py\", line 61, in generate\n    return worker.predict(\n  File \"<MULTI_MODEL_NOTEBOOK_DIR>/app/worker.py\", line 100, in predict\n    raise ValueError(\"Empty prompt\")\nValueError: Empty prompt\n"}}


## Predict string input

In [12]:
with httpx.Client(trust_env=False, proxy=None) as client:
    response = client.post(
        f"{BASE}/predict",
        json={"model_id": TINY, "input_data": "Say hi in three words."},
        timeout=180.0,
    )

print("status:", response.status_code)
print(response.text)
if response.status_code == 200:
    data = response.json()
    assert data["model_id"] == TINY, f"routing mismatch: {data['model_id']}"
    print("routing OK:", data["model_id"])

status: 200
{"model_id":"tinyllama-1.1b","model_name":"TinyLlama-1.1B-Chat-v1.0","prompt":"Say hi in three words.","generated_text":"<|user|>\nWrite a step-by-step guide on how to create a DIY home decor project that will add personality and charm to your space. Include details on the materials needed, the tools required, the design process, and any tips or tricks for achieving a polished outcome"}
routing OK: tinyllama-1.1b


## Cold start

In [15]:
import time

with httpx.Client(trust_env=False, proxy=None) as client:
    cleared = client.post(f"{BASE}/admin/cache/clear", timeout=60.0)
    print("clear:", cleared.status_code, cleared.json())

    models = client.get(f"{BASE}/models", timeout=30.0).json()
    print("loaded after clear:", models["loaded_models"])

    times = []
    for i in range(2):
        start = time.perf_counter()
        r = client.post(
            f"{BASE}/generate",
            json={
                "model_id": QWEN,
                "prompt": "What is KV cache? One sentence.",
                "max_new_tokens": 32,
            },
            timeout=300.0,
        )
        elapsed = time.perf_counter() - start
        times.append(elapsed)
        label = "COLD (load+gen)" if i == 0 else "WARM (gen only)"
        print(f"{label}: {round(elapsed, 3)}s  status={r.status_code}")

    print("cold/warm ratio:", round(times[0] / times[1], 2) if times[1] else "n/a")

clear: 404 {'detail': 'Not Found'}
loaded after clear: {'tinyllama-1.1b': 'TinyLlama-1.1B-Chat-v1.0', 'qwen2.5-0.5b': 'Qwen2.5-0.5B-Instruct'}
COLD (load+gen): 1.012s  status=200
WARM (gen only): 0.949s  status=200
cold/warm ratio: 1.07


## LRU eviction

In [ ]:
import time

with httpx.Client(trust_env=False, proxy=None) as client:
    # Cap cache at 1 model
    cfg = client.post(
        f"{BASE}/admin/cache/config",
        json={"max_models": 1},
        timeout=60.0,
    )
    print("config:", cfg.status_code, cfg.json())

    client.post(f"{BASE}/admin/cache/clear", timeout=60.0)

    sequence = [QWEN, TINY, QWEN, TINY]
    for mid in sequence:
        start = time.perf_counter()
        r = client.post(
            f"{BASE}/generate",
            json={"model_id": mid, "prompt": "Say hello.", "max_new_tokens": 16},
            timeout=300.0,
        )
        elapsed = time.perf_counter() - start
        loaded = client.get(f"{BASE}/models", timeout=30.0).json()["loaded_models"]
        print(
            f"request={mid}  {round(elapsed, 3)}s  status={r.status_code}  "
            f"loaded={list(loaded.keys())}"
        )
        assert list(loaded.keys()) == [mid], f"expected only {mid} loaded, got {loaded}"

    # Restore capacity for later cells
    restored = client.post(
        f"{BASE}/admin/cache/config",
        json={"max_models": 2},
        timeout=60.0,
    )
    print("restored:", restored.json())

## Concurrent same model

In [18]:
import asyncio
import time

prompts = [
    "What is attention? One sentence.",
    "What is a transformer? One sentence.",
    "What is KV cache? One sentence.",
    "What is batching? One sentence.",
]


async def call_qwen(prompt: str):
    async with httpx.AsyncClient(trust_env=False, proxy=None) as client:
        start = time.perf_counter()
        r = await client.post(
            f"{BASE}/generate",
            json={"model_id": QWEN, "prompt": prompt, "max_new_tokens": 32},
            timeout=300.0,
        )
        return {
            "prompt": prompt,
            "time_s": round(time.perf_counter() - start, 2),
            "status": r.status_code,
            "model_id": r.json().get("model_id") if r.status_code == 200 else None,
        }


results = await asyncio.gather(*[call_qwen(p) for p in prompts])
for r in results:
    print(r)
    assert r["status"] == 200
    assert r["model_id"] == QWEN

{'prompt': 'What is attention? One sentence.', 'time_s': 2.71, 'status': 200, 'model_id': 'qwen2.5-0.5b'}
{'prompt': 'What is a transformer? One sentence.', 'time_s': 2.87, 'status': 200, 'model_id': 'qwen2.5-0.5b'}
{'prompt': 'What is KV cache? One sentence.', 'time_s': 2.76, 'status': 200, 'model_id': 'qwen2.5-0.5b'}
{'prompt': 'What is batching? One sentence.', 'time_s': 2.29, 'status': 200, 'model_id': 'qwen2.5-0.5b'}


## Token latency

In [19]:
import time

prompt = "Explain continuous batching briefly."
token_counts = [8, 32, 64, 128]

with httpx.Client(trust_env=False, proxy=None) as client:
    # warm once
    client.post(
        f"{BASE}/generate",
        json={"model_id": QWEN, "prompt": prompt, "max_new_tokens": 8},
        timeout=180.0,
    )

    for n in token_counts:
        start = time.perf_counter()
        r = client.post(
            f"{BASE}/generate",
            json={"model_id": QWEN, "prompt": prompt, "max_new_tokens": n},
            timeout=300.0,
        )
        elapsed = time.perf_counter() - start
        text = r.json().get("generated_text", "") if r.status_code == 200 else r.text
        print(f"max_new_tokens={n:3d}  {round(elapsed, 3)}s  chars={len(text)}  status={r.status_code}")

max_new_tokens=  8  0.28s  chars=59  status=200
max_new_tokens= 32  0.953s  chars=197  status=200
max_new_tokens= 64  1.896s  chars=361  status=200
max_new_tokens=128  3.788s  chars=765  status=200
